# 09. The rolling evaluation: the 24 paths, the frontier, E4's second clause, E5 and E6

**What this notebook does.** It reads the records the script `scripts/run_evaluation.py` wrote on this machine, one row per path and month, and computes each path's measures over the evaluation window, July 1999 to December 2024, and by year. It judges E4's second clause, E5 and E6, and writes the tables that are committed.

**Why it comes here.** The script runs the paths, which takes hours. This notebook only reads what it wrote, so it can be run again at once. Nothing by security leaves `outputs/`, which git ignores.

**Terms used in this notebook.**

- *market capitalisation*: price times shares outstanding, at the end of the month before.
- *benchmark*: the universe weighted by market capitalisation at the start of the month.
- *active weight*: a stock's weight in the portfolio minus its weight in the benchmark. The *active return* is the portfolio's return minus the benchmark's in the month.
- *covariance matrix*: the table of the variances of the members' monthly returns on its diagonal and their covariances off it.
- *factor*: a common source of return, here French's five factors and his momentum factor.
- *covariance estimator*: a rule that builds the covariance matrix from three years of daily returns. Notebook 07 has four: the sample estimator, the Ledoit-Wolf estimator, the factor estimator, which explains each stock's returns by the factors, and the PCA estimator.
- *turnover*: here one-way turnover, half the sum of the absolute changes in weight, the share of the portfolio sold and bought.
- *turnover cap*: the limit on the turnover the optimiser chooses, measured from the reference portfolio, last month's portfolio after its returns with the stocks that left sold and the entrants bought at their benchmark weight. *Forced turnover* is that move. A cap *binds* when the month's chosen turnover is at the cap, and a month is *relaxed* when the cap had to be raised to the smallest feasible turnover.
- *constraint set*: C0 is long-only. C1 adds bounds per stock, per company and per sector, C2 the turnover cap, and C3, the *mandate*, the *tilt*, which holds every Energy and Utilities stock at no more than half its benchmark weight, the exclusion of two sub-industries and a *beta*, the slope of the portfolio's return on the benchmark's, equal to one. Notebook 08 sets them out.
- *path*: one specification run month by month over the evaluation window: a covariance estimator under a constraint set at a turnover cap. There are 24: the four estimators under the four sets C0 to C3, at the 2 per cent cap where the set has a cap, and the Ledoit-Wolf and factor estimators under the mandate at caps of 0.5, 1 and 4 per cent and with no cap. Every path starts as the benchmark.
- *tracking error*: the standard deviation of the monthly active returns, times the square root of 12. The *realised tracking error* is measured on the returns, and the *forecast tracking error* is the one the month's covariance matrix implied when the weights were chosen.
- *bias statistic*: the standard deviation of each month's active return divided by that month's forecast monthly tracking error. It is one when the forecasts are right on average, above one when they are too low. It is taken over the months whose forecast tracking error is above one millionth a year, and the number of months it rests on is given beside it.
- *frontier*: the forecast and realised tracking error of one estimator under the mandate at five caps, from the tightest to none.
- *basis point*: a hundredth of a percentage point.
- *cost*: 10 basis points per unit of total turnover, forced included, and 20 as the sensitivity.
- *Newey-West standard error*: the standard error of a mean that allows each month to be correlated with the next twelve.
- *cell rule*: a cell of the table by month is written when at least three stocks have an active weight above one millionth and no stock's absolute contribution to the active return is more than 80 per cent of the sum of the absolute contributions. A cell that fails is left empty and counted.
- *provenance*: the record of what a result was built from.
- *verdict*: the record of whether an expectation fixed before the run is met.

**What I expect to see, fixed in `src/slt/constants.py` before any path ran.**

- *E4, second clause*: under the mandate at the 2 per cent cap, the sample estimator's realised tracking error is above the Ledoit-Wolf estimator's.
- *E5*: for the Ledoit-Wolf and the factor estimators, the mean forecast tracking error does not rise as the cap loosens from 0.5 per cent to none, and the realised tracking error at the 4 per cent cap is below that at 0.5 per cent.
- *E6*: under the mandate at the 2 per cent cap, the realised tracking error of the Ledoit-Wolf, factor and PCA estimators each lies between 50 and 150 basis points a year, and each one's bias statistic between 0.8 and 1.25.
- Under C0, C1 and C2 the portfolio is the benchmark in every month, to within one millionth of a weight.

Running time: under a minute, once the script has run.

In [ ]:
import json
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X

root = X.data_root()             # stops with one sentence when the data root is not set
manifest = X.check_manifest(root)   # stops when the manifest on disk is not the one the constants fix
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}")

In [ ]:
import numpy as np

from slt import evaluation as V

OUT = ROOT / C.OUTPUTS_DIR / "evaluation"

## 1. The script's run

The provenance record the script wrote: when it started and finished, the versions of the solver and its libraries, and the threads of linear algebra in each process. Then the records of every path.

In [ ]:
prov = json.loads((OUT / "provenance.json").read_text())
print(json.dumps(prov, indent=1))
assert not any(prov["exit_codes"].values()), "a group of the script stopped with an error: run the script again to resume"
records = V.load_records(OUT)
missing = [C.PATH_LABELS[p] for p in C.PATHS if C.PATH_LABELS[p] not in records]
assert not missing, f"no records for {missing}: run the script to its end first"
for lab, rec in records.items():
    assert len(rec) == C.EVAL_MONTHS, f"{lab} has {len(rec)} months"
print(f"{len(records)} paths, {C.EVAL_MONTHS} months each")

## 2. Every path over the window

In [ ]:
summaries = {lab: V.summarise(rec) for lab, rec in records.items()}
show = pd.DataFrame({lab: {"realised TE (bp)": s["realised_te"] * 1e4, "forecast TE (bp)": s["forecast_te_mean"] * 1e4,
                           "bias": s["bias"], "bias months": s["bias_months"], "active gross (bp)": s["active_return_gross"] * 1e4,
                           "net at 10 bp (bp)": s["active_return_net_at_10bp"] * 1e4, "turnover / yr": s["turnover_total_per_year"],
                           "cap binds": s["cap_binding_share"], "relaxed": s["relaxed_months"], "held": s["held_mean"]}
                     for lab, s in summaries.items()}).T
print(show.round(3).to_string())

## 3. C0, C1 and C2: the benchmark

Under these sets nothing pushes the portfolio off the benchmark. The months in which it differs from the benchmark by more than one millionth of a weight, and the largest absolute active weight in any month:

In [ ]:
for p in C.PATHS:
    if p[1] != C.MANDATE:
        s = summaries[C.PATH_LABELS[p]]
        print(f"{C.PATH_LABELS[p]}: months off the benchmark {s['months_off_benchmark']} | largest absolute active weight {s['largest_abs_active_weight']:.2e}")

## 4. The frontier, E4's second clause, E5 and E6

In [ ]:
verdicts = V.judge(summaries)
for est in C.FRONTIER_ESTIMATORS:
    v = verdicts[f"E5_{est}"]
    caps = ["none" if c is None else f"{100 * c:g}%" for c in v["caps"]]
    print(est, "mean forecast TE (bp) by cap:", dict(zip(caps, [round(x * 1e4, 2) for x in v["forecast_te_mean"]])))
    print(f"E5, {est}: {'met' if v['met'] else 'not met'}. The mean forecast tracking error "
          f"{'does not rise' if v['forecast_nonincreasing'] else 'rises'} as the cap loosens, and the realised tracking error is "
          f"{v['realised_te_looser'] * 1e4:.1f} basis points a year at the 4 per cent cap against {v['realised_te_tighter'] * 1e4:.1f} at 0.5 per cent.")
NAMES = {"sample": "sample", "ledoit_wolf": "Ledoit-Wolf", "factor": "factor", "pca": "PCA"}
def the(ests):
    names = [f"the {NAMES[e]}" for e in ests]
    return (", ".join(names[:-1]) + " and " + names[-1] if len(names) > 1 else names[0]) + (" estimators" if len(names) > 1 else " estimator")
def one_line(name):
    v = verdicts[name]
    parts = ([f"holds for {the(v['holds_for'])}"] if v["holds_for"] else []) + ([f"fails for {the(v['fails_for'])}"] if v["fails_for"] else [])
    return f"{name}: {'met' if v['met'] else 'not met'}. It " + " and ".join(parts) + "."
print(one_line("E5"))
v = verdicts["E4_second_clause"]
print(f"E4, second clause: {'met' if v['met'] else 'not met'}. Under the mandate at the 2 per cent cap the sample estimator's realised "
      f"tracking error is {v['sample_realised_te'] * 1e4:.1f} basis points a year against Ledoit-Wolf's {v['ledoit_wolf_realised_te'] * 1e4:.1f}.")
for est in ("ledoit_wolf", "factor", "pca"):
    v = verdicts[f"E6_{est}"]
    print(f"E6, {est}: {'met' if v['met'] else 'not met'}. Realised tracking error {v['realised_te'] * 1e4:.1f} basis points a year "
          f"against 50 to 150, bias statistic {v['bias']:.3f} over {v['bias_months']} months against 0.8 to 1.25.")
print(one_line("E6"))

## 5. What is committed

Five files go into `results/20260928/`. The active returns by month of the twelve paths under the mandate, gross of costs, each cell under the cell rule; the measures of every path over the window and by year, with no weight and no name; the verdicts of milestone 3, E7 added by notebook 10; and the provenance of the script's run.

In [ ]:
table, empty = V.active_returns_table(records)
by_year = pd.concat([V.by_year(rec).assign(path=lab) for lab, rec in records.items()], ignore_index=True)
by_year = by_year[["path"] + [c for c in by_year.columns if c != "path"]]
withheld = [C.PATH_LABELS[p] for p in C.PATHS if p[1] not in C.LARGEST_ACTIVE_WEIGHT_SETS]   # one stock's weight under C3
by_year.loc[by_year["path"].isin(withheld), "largest_abs_active_weight"] = float("nan")
committed = {lab: dict(summaries[lab], **({"largest_abs_active_weight": None} if lab in withheld else {})) for lab in records}
summary = V.jsonable({"written_by": "notebooks/09_rolling_evaluation.ipynb", "manifest_sha256": C.MANIFEST_SHA256,
                      "paths": committed, "empty_cells_by_column": empty,
                      "costs": {"base": C.COST_PER_UNIT_TURNOVER, "sensitivity": C.COST_SENSITIVITY, "on": C.COST_BASE}})
path = ROOT / C.RESULTS_DIR / "verdicts_milestone3.json"
earlier = json.loads(path.read_text()) if path.exists() else {}
record = V.jsonable({k: v for k, v in verdicts.items() if k != "E7"})
if "E7" in earlier:
    record["E7"] = earlier["E7"]                     # written by notebook 10
print("empty cells by column:", empty)
for p in [X.write_results("active_returns_monthly.csv", frame=table),
          X.write_results("paths_summary.json", record=summary),
          X.write_results("paths_by_year.csv", frame=by_year),
          X.write_results("verdicts_milestone3.json", record=record),
          X.write_results("provenance_evaluation.json", record=prov)]:
    print("written:", p.relative_to(ROOT))

## What was learned, and what could be wrong

What the notebook establishes: each path's realised and forecast tracking error, the forecasts' calibration, the cost of the turnover each path chose, and the verdicts of E4's second clause, E5 and E6.

What could still be wrong. A realised tracking error over 306 months has its own sampling error, about 4 per cent of its value. A bias statistic near one over the window can hide periods in which the forecasts were too low and others in which they were too high. The costs are a flat rate per unit of turnover, with no market impact. And every path's active return is measured against this study's benchmark, which weights the members by their full market capitalisation. The published index weights them by the shares available to the public, so the two differ.